In [6]:
import pandas as pd
import numpy as np

TIME_OFFSET = 10800 #ADT to UTC is 3hrs

In [8]:
def process_file(input_csv, output_csv):
    print(f"Processing {input_csv}...")

    df = pd.read_csv(input_csv)

    # Convert timestamp to UTC
    df["timestamp"] = df["frame.time_epoch"] + TIME_OFFSET

    # Sort for IAT calculation
    df = df.sort_values("timestamp")

    # Create 1-second window
    df["time_window"] = df["timestamp"].astype(int)

    # Calculate inter-arrival time
    df["iat"] = df.groupby(["ip.src", "ip.dst"])["timestamp"].diff().fillna(0)

    # Aggregate features
    flows = df.groupby(
        ["time_window", "ip.src", "ip.dst"]
    ).agg(
        packet_count=("frame.len", "count"),
        total_bytes=("frame.len", "sum"),
        mean_packet_size=("frame.len", "mean"),
        std_packet_size=("frame.len", "std"),
        iat_mean=("iat", "mean"),
        iat_std=("iat", "std"),
        unique_func_codes=("modbus.func_code", "nunique"),
        exception_count=("modbus.exception_code", lambda x: x.notna().sum())
    ).reset_index()

    flows = flows.fillna(0)

    flows.to_csv(output_csv, index=False)
    print(f"Saved → {output_csv}\n")

In [13]:
process_file("../train/benign_nw.csv", "../train/benign_flows.csv")

Processing ../train/benign_nw.csv...
Saved → ../train/benign_flows.csv



In [14]:
process_file("../train/cscada_attack_ssw.csv", "../train/cscada_flows.csv")

Processing ../train/cscada_attack_ssw.csv...
Saved → ../train/cscada_flows.csv



In [15]:
process_file("../train/ext_attack_nw.csv", "../train/external_flows.csv")

Processing ../train/ext_attack_nw.csv...
Saved → ../train/external_flows.csv

